# **Replication File for Hoshino and Pancs (2026)**

**Paper Title:** Estimating Pair-Specific Network Effects in Binary-Action Games: Two-Sided Markets via Restricted Boltzmann Machines

**Journal:** Management Science

**Author:** Tetsuya Hoshino

**Original Code Last Updated:** September 22, 2024

**Documentation Last Updated:** September 24, 2026

**Update Note:** All code cells are unchanged from the original notebook.

**Description:** This notebook contains the replication code for the numerical figures in the paper named above.

**GPU requirement:** For estimation, select an **NVIDIA T4 GPU** in Google Colab: **Runtime > Change runtime type > T4 GPU**. A GPU is required for practical computation; each method runs for about two hours. Estimation has been tested only in Google Colab.

**Link to Paper:** The paper can be found at [this website](https://www.tetsuyahoshino.com).

**Acknowledgment:** Romans Pancs and I thank Natalia Denisenko for her excellent research assistance.


## **Preliminaries**

#### **Libraries and GPU Setup**

We import necessary libraries and set up GPU usage if available.

In [ ]:
# Standard library imports
import glob
import itertools
import json
import random
import time

# Third-party imports
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader

# Google Colab specific imports
from google.colab import files

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#### **Configuration**

We define the parameters for our datasets and training process.

In [ ]:
class Config:
    def __init__(self):
        # RBM parameters
        self.n1 = 11  # Number of nodes on layer 1
        self.n2 = 11  # Number of nodes on layer 2

        self.set_rand_seed()
        self.true_W = torch.randn(self.n1, self.n2).to(device)
        self.true_b1 = torch.randn(self.n1).to(device)
        self.true_b2 = torch.randn(self.n2).to(device)

        # Data generation parameters
        self.data_size = 100_000
        self.batch_size = 1_000
        self.gibbs_step = 500

        # Training parameters
        self.time_limit = 7200  # in seconds
        self.cd_step = 1 # only for Contrastive Divergence

        # Learning rates
        self.learning_rates = [0.1]

        # Training methods
        self.training_methods = ['maximum_likelihood', 'contrastive_divergence']

        # File name
        self.data_filename = f"synthetic_data_N{self.n1}x{self.n2}_D{self.data_size}_B{self.batch_size}.json"

    def get_results_filename(self, training_method):
        training_method_abbr = 'cd' if training_method == 'contrastive_divergence' else 'ml'
        return f"training_data_N{self.n1}x{self.n2}_D{self.data_size}_B{self.batch_size}_{training_method_abbr}.json"

    @staticmethod
    def set_rand_seed(seed=42):
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

# Create a config instance
config = Config()
print(f"Configuration initialized. Using device: {device}")

## **Fully Visible Restricted Boltzmann Machine**

#### **Model**

Our study formally establishes the equivalence between:

*   **Two-sided markets with heterogeneous network externalities**
*   **Fully visible restricted Boltzmann machine (FVRBM)**

This equivalence allows us to estimate the two-sided market model using machine-learning techniques to train the FVRBM. In particular, the powerful training algorithm called Contrastive Divergence enables us to estimate all two-sided market parameters consistently while mitigating computational intractability. See our paper for details.

The **FVRBM** is defined as follows:

- $N_1$ is the number of nodes in layer 1
- $N_2$ is the number of nodes in layer 2
- $\mathbf{x}_1 \in \{0, 1\}^{N_1}$ is the vector of layer-1 nodes
- $\mathbf{x}_2 \in \{0, 1\}^{N_2}$ is the vector of layer-2 nodes
    - $\mathbf{x} = (\mathbf{x}_1,\mathbf{x}_2)$ is the vector of all nodes

The energy function of the FVRBM is defined as:

$$E(\mathbf{x}) = - \mathbf{b}_1^{\top} \mathbf{x}_1 - \mathbf{b}_2^{\top} \mathbf{x}_2 - \mathbf{x}_1^{\top} \mathbf{W} \mathbf{x}_2,$$

where:
- $\mathbf{W} \in \mathbb{R}^{N_1 \times N_2}$ is the weight matrix
- $\mathbf{b}_1 \in \mathbb{R}^{N_1}$ is the bias vector for layer 1
- $\mathbf{b}_2 \in \mathbb{R}^{N_2}$ is the bias vector for layer 2

The joint probability distribution is given by the Boltzmann distribution:

$$\mathbb{P}(\mathbf{x}) = \frac{1}{Z} \exp\{-E(\mathbf{x})\},$$

where $Z$ is the normalizing constant, also called the partition function, defined as:

$$Z = \sum_{\mathbf{x}'} \exp\{-E(\mathbf{x}')\}.$$

#### **Training Algorithms**

Here we introduce two training algorithms for the FVRBM:
1.   The **Maximum Likelihood** method and its computational intractability
2.   The **Contrastive Divergence** method ([Hinton 2002](https://ieeexplore.ieee.org/abstract/document/6789337))


The **Maximum Likelihood (ML) method** maximizes the log likelihood function defined by

$$L(\mathbf{W},\mathbf{b}) \equiv \frac{1}{\mathsf{data\_size}} \sum_{t} \log \mathbb{P}(\mathbf{x}(t); \mathbf{W},\mathbf{b}),$$

which is strictly concave for FVRBMs, with a unique maximizer on the paper's parameter set. (This argument does not apply to RBMs with hidden layers.)

To find the maximizer, we use the **gradient ascent method**, which updates the parameters according to:

$$
\begin{aligned}
\mathbf{W}^{(t+1)} &= \mathbf{W}^{(t)} + \eta \nabla_{\mathbf{W}} L,\\
\mathbf{b}^{(t+1)} &= \mathbf{b}^{(t)} + \eta \nabla_{\mathbf{b}} L,
\end{aligned}
$$

where $\eta$ denotes the learning rate.
By direct calculation,

$$
\begin{aligned}
\nabla_{\mathbf{W}} L &=
\color{blue}{
    \frac{1}{T} \sum_{t=1}^{T} \mathbf{x}_{1}(t)\mathbf{x}_{2}^{\top}(t)
} -
\color{red}{
    \sum_{\mathbf{x}\in\{0,1\}^{n}}\mathbb{P}\left(\mathbf{x}\mid\mathbf{W},\mathbf{b}\right)\mathbf{x}_{1}\mathbf{x}_{2}^{\top}
}, \\
\nabla_{\mathbf{b}} L &=
\color{blue}{
    \frac{1}{T} \sum_{t=1}^{T}\mathbf{x}(t)
} \qquad\,-
\color{red}{
    \sum_{\mathbf{x}\in\{0,1\}^{n}}\mathbb{P}\left(\mathbf{x}\mid\mathbf{W},\mathbf{b}\right)\mathbf{x}
},
\end{aligned}
$$

where

*   each term in blue is called the **positive phase**;
*   each term in red is called the **negative phase**.

The computation of the positive phase is straightforward; one only needs to compute the sample average from the data.

By contrast, the computation of the negative phase is computationally difficult. This is because this computation involves summing $2^n$ terms, a computationally costly task. For example, if $n=20$ then we have $2^{20} \approx 1,000,000$ terms, and if $n=30$ then we have $2^{30} \approx 1,000,000,000$ terms. In other words, this summation suffers from **combinatorial explosion**.

Indeed, the ML method computes the negative phase by tackling this computationally intractable sum. This approach works well *only* when $n$ is small.

<br>

The **Contrastive Divergence (CD) method** approximates the negative phase: Given a CD-step $k=1,2,\ldots$,

$$
\begin{aligned}
\sum_{\mathbf{x}\in\{0,1\}^{n}}\mathbb{P}\left(\mathbf{x}\mid\mathbf{W},\mathbf{b}\right)\mathbf{x}_{1}\mathbf{x}_{2}^{\top} &\approx \frac{1}{T} \sum_{t=1}^{T} \mathbf{\tilde{x}}_{1}^{(k)}(t) \mathbf{\tilde{x}}_{2}^{(k)\top}(t),\\
\sum_{\mathbf{x}\in\{0,1\}^{n}}\mathbb{P}\left(\mathbf{x}\mid\mathbf{W},\mathbf{b}\right)\mathbf{x} &\approx \frac{1}{T} \sum_{t=1}^{T} \mathbf{\tilde{x}}^{(k)}(t),
\end{aligned}
$$

where $\mathbf{\tilde{x}}^{(k)}(t) = (\mathbf{\tilde{x}}_{1}^{(k)}(t),\mathbf{\tilde{x}}_{2}^{(k)}(t))$ is generated by the following **blocked Gibbs sampler**:

0. Let $\mathbf{x}^{(0)}(t) \equiv \mathbf{x}(t)$ be the initial data point.
1. Sample $\mathbf{x}_{1}^{(1)}(t) \sim \mathbb{P}(\cdot \mid \mathbf{x}_{2}^{(0)}(t); \mathbf{W},\mathbf{b})$ and then $\mathbf{x}_{2}^{(1)}(t) \sim \mathbb{P}(\cdot \mid \mathbf{x}_{1}^{(1)}(t); \mathbf{W},\mathbf{b})$.
2. Sample $\mathbf{x}_{1}^{(2)}(t) \sim \mathbb{P}(\cdot \mid \mathbf{x}_{2}^{(1)}(t); \mathbf{W},\mathbf{b})$ and then $\mathbf{x}_{2}^{(2)}(t) \sim \mathbb{P}(\cdot \mid \mathbf{x}_{1}^{(2)}(t); \mathbf{W},\mathbf{b})$, and so forth.

Gibbs chains are independent across observations; successive steps within a chain are dependent.

The **CD method iterates the Gibbs sampler only for a small $k$ number of times --- typically only for $k=1$, that is, once!** Naively, one would have thought that CD would not work well and surely would not deliver a consistent estimator. This is because the Gibbs sampler needs a large number of steps $k$ to (almost) converge. The approximation with k=1 is heavily biased --- as is the gradient $\nabla_{\mathbf{W}} L$ and $\nabla_{\mathbf{b}} L$.

Perhaps surprisingly, the CD method has achieved empirical success in machine-learning contexts ([Carreira-Perpiñán and Hinton 2005](https://proceedings.mlr.press/r5/carreira-perpinan05a.html); [Bengio and Delalleau 2009](https://ieeexplore.ieee.org/abstract/document/6795523)). Our consistency result concerns fully visible RBMs.

For our two-sided market model, however, we demonstrate that the CD method provides a consistent estimator. Our argument makes use of a recent result by [Jiang, Wu, Jin, and Wong 2018](https://projecteuclid.org/journals/annals-of-statistics/volume-46/issue-6A/Convergence-of-contrastive-divergence-algorithm-in-exponential-family/10.1214/17-AOS1649.full). Consistency means that as we increase the sample size, CD parameter estimates converge to the underlying true values. As a result, the use of CD in our context is theoretically justified.

**Remark:** If we take the number of iterations $k$ in CD large enough, the Gibbs chain approaches its stationary distribution, providing an accurate approximation of the negative phase. In this case, however, Gibbs sampling is impractically slow. Thus, using CD with a large $k$ does not resolve computational intractability.

#### **Implementation of FVRBM and Training Algorithms**

In [ ]:
class RBM(nn.Module):
    def __init__(self, config):
        super(RBM, self).__init__()
        self.config = config
        self.n1 = config.n1  # Number of nodes on layer 1
        self.n2 = config.n2  # Number of nodes on layer 2

        self.cd_step = config.cd_step  # Number of steps in contrastive divergence

        # Initialize model parameters
        self.W = nn.Parameter(torch.randn(self.n1, self.n2).to(device))  # Weight matrix
        self.b1 = nn.Parameter(torch.randn(self.n1).to(device))  # Bias for layer 1
        self.b2 = nn.Parameter(torch.randn(self.n2).to(device))  # Bias for layer 2

    def maximum_likelihood(self, data1, data2):
        """Perform maximum likelihood estimation."""
        # Positive phase
        positive_phase = data1.T @ data2

        # Generate all possible binary states
        self.states1 = torch.tensor(list(itertools.product([0, 1], repeat=self.n1)), dtype=torch.float).to(device)
        self.states2 = torch.tensor(list(itertools.product([0, 1], repeat=self.n2)), dtype=torch.float).to(device)

        # Prepare tensors for efficient computation
        states1_expanded = self.states1.unsqueeze(2).expand(-1, -1, self.n2).unsqueeze(1)
        states2_expanded = self.states2.unsqueeze(1).expand(-1, self.n1, -1).unsqueeze(0)
        self.all_states = (states1_expanded * states2_expanded).reshape(-1, self.n1, self.n2)

        # Compute energies for all states
        all_W_contributions = torch.einsum('bij,ij->bij', self.all_states, self.W)
        all_b1_contributions = torch.repeat_interleave(torch.einsum('bi,i->bi', self.states1, self.b1), self.states2.shape[0], dim=0)
        all_b2_contributions = torch.einsum('bj,j->bj', self.states2, self.b2).repeat(self.states1.shape[0], 1)
        energies = -(torch.sum(all_W_contributions, (1, 2)) + torch.sum(all_b1_contributions, 1) + torch.sum(all_b2_contributions, 1))

        # Compute the partition function
        probabilities = torch.exp(-energies)
        partition_function = probabilities.sum()

        # Compute expectations
        expected_W_products = torch.mul(probabilities[:, None, None].expand(-1, self.W.shape[0], self.W.shape[1]), self.all_states)
        expected_b1_activations = torch.mul(probabilities[:, None].expand(-1, self.W.shape[0]), torch.repeat_interleave(self.states1, self.states2.shape[0], dim=0))
        expected_b2_activations = torch.mul(probabilities[:, None].expand(-1, self.W.shape[1]), self.states2.repeat(self.states1.shape[0], 1))

        negative_phase = expected_W_products.sum(0) / partition_function
        expected_data1 = expected_b1_activations.sum(0) / partition_function
        expected_data2 = expected_b2_activations.sum(0) / partition_function

        # Compute gradients
        self.W.grad = -(positive_phase / len(data1) - negative_phase)
        self.b1.grad = -torch.mean(data1 - expected_data1.squeeze(), dim=0)
        self.b2.grad = -torch.mean(data2 - expected_data2.squeeze(), dim=0)

    def contrastive_divergence(self, data1, data2):
        """Perform contrastive divergence."""
        # Positive phase
        positive_phase = data1.T @ data2

        # Negative phase
        new_data1, new_data2 = self.perform_gibbs_sampling(data2.clone())
        negative_phase = new_data1.T @ new_data2

        # Compute gradients
        self.W.grad = -(positive_phase - negative_phase) / len(data1)
        self.b1.grad = -torch.mean(data1 - new_data1, dim=0)
        self.b2.grad = -torch.mean(data2 - new_data2, dim=0)

    def perform_gibbs_sampling(self, data2):
        """Perform Gibbs sampling."""
        with torch.no_grad():
            for _ in range(self.cd_step):
                data1 = torch.bernoulli(torch.sigmoid(data2 @ self.W.T + self.b1))
                data2 = torch.bernoulli(torch.sigmoid(data1 @ self.W + self.b2))

        return data1, data2

# RBM instance
model = RBM(config)
print(f"RBM initialized with {config.n1}x{config.n2} architecture")

## **Generating or Loading Synthetic Data**

We generate synthetic data or load it for training.

In [ ]:
class MyDataset:
    def __init__(self, config):
        self.config = config

        self.data_size = config.data_size
        self.gibbs_step = config.gibbs_step

        self.true_W  = config.true_W
        self.true_b1 = config.true_b1
        self.true_b2 = config.true_b2

    def generate_data(self):
        """Generate synthetic data using Gibbs sampling."""
        # Initialize random data
        data1 = torch.bernoulli(torch.rand(self.data_size, self.config.n1)).to(device)
        data2 = torch.zeros(self.data_size, self.config.n2).to(device)

        # Perform Gibbs sampling
        for _ in tqdm(range(self.gibbs_step), desc='Generating data', leave=True):
            data2 = torch.bernoulli(torch.sigmoid(data1 @ self.true_W + self.true_b2))
            data1 = torch.bernoulli(torch.sigmoid(data2 @ self.true_W.T + self.true_b1))

        return data1, data2

    def save_data(self, data1, data2, filename):
        """Save generated data to a file."""
        print(f"Saving data to {filename}")

        data_to_save = {
            "data_parameters": {
                "n1" : self.config.n1,
                "n2" : self.config.n2,
                "data_size" : self.data_size,
                "gibbs_step" : self.gibbs_step
            },
            "true_parameters": {
                "W"  : self.true_W.cpu().tolist(),
                "b1" : self.true_b1.cpu().tolist(),
                "b2" : self.true_b2.cpu().tolist()
            },
            "data1" : data1.cpu().tolist(),
            "data2" : data2.cpu().tolist()
        }

        with open(filename, 'w') as f:
            json.dump(data_to_save, f)

        print(f"Data saved to {filename}")
        files.download(filename)

    @staticmethod
    def load_data(filename, config):
        """Load data from a file."""
        print(f"Loading data from {filename}")

        with open(filename, 'r') as f:
            loaded_data = json.load(f)

        data1 = torch.tensor(loaded_data['data1']).to(device)
        data2 = torch.tensor(loaded_data['data2']).to(device)

        loaded_true_params = {k: torch.tensor(v).to(device) for k, v in loaded_data['true_parameters'].items()}

        return data1, data2, loaded_true_params

# MyDataset instance
my_dataset = MyDataset(config)

# Check if data file exists
import os

if not os.path.exists(config.data_filename):
    # Generate and save new data
    print("Generating new data...")
    data1, data2 = my_dataset.generate_data()
    my_dataset.save_data(data1, data2, config.data_filename)
else:
    print("Loading existing data...")
    data1, data2, loaded_true_params = MyDataset.load_data(config.data_filename, config)

# Create DataLoader
dataset = TensorDataset(data1, data2)
dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)
print(f"Data loaded. Shape: data1 {data1.shape}, data2 {data2.shape}")

## **Training and Evaluation**

We train the FVRBM and evaluate the performance of the training algorithms.

In [ ]:
class Trainer:
    def __init__(self, config, model, dataloader):
        self.config = config
        self.model = model

        self.dataloader = dataloader

        self.initial_params = None

    def compute_loss(self):
        """Compute Mean Squared Error between true and estimated parameters."""
        def compute_mse(a, b):
            return ((a - b) ** 2).mean().item()

        model_params = {
            'W'  : self.model.W.detach().cpu(),
            'b1' : self.model.b1.detach().cpu(),
            'b2' : self.model.b2.detach().cpu()
        }

        losses = {
            f'mse_{name}': compute_mse(getattr(self.config, f'true_{name}').cpu(), model_params[name])
            for name in ['W', 'b1', 'b2']
        }

        all_true = torch.cat([getattr(self.config, f'true_{name}').cpu().flatten() for name in ['W', 'b1', 'b2']])
        all_model = torch.cat([p.flatten() for p in model_params.values()])
        losses['mse_Wb'] = compute_mse(all_true, all_model)

        return losses

    def save_initial_params(self):
        """Save the initial parameters of the model."""
        self.initial_params = {
            'W'  : self.model.W.clone().detach(),
            'b1' : self.model.b1.clone().detach(),
            'b2' : self.model.b2.clone().detach()
        }

    def reset_to_initial_params(self):
        """Reset the model to the initial parameters."""
        with torch.no_grad():
            self.model.W.copy_(self.initial_params['W'])
            self.model.b1.copy_(self.initial_params['b1'])
            self.model.b2.copy_(self.initial_params['b2'])

    def train(self, training_method, learning_rate):
        """Train the RBM model."""
        self.model.train()

        # Reset to initial parameters
        if self.initial_params is None:
            self.save_initial_params()
        else:
            self.reset_to_initial_params()

        # Set up optimizer with the current learning rate
        optimizer = optim.SGD(self.model.parameters(), lr=learning_rate)

        losses = []

        # Record initial loss
        initial_loss = self.compute_loss()
        initial_loss.update({'time': 0, 'epoch': 0, 'avg_epoch_loss': initial_loss['mse_Wb']})
        losses.append(initial_loss)

        start_time = time.time()
        epoch = 0

        while time.time() - start_time < self.config.time_limit:
            epoch_loss = 0
            for data1, data2 in tqdm(self.dataloader, desc=f'Epoch {epoch + 1}', leave=False):
                data1, data2 = data1.to(device), data2.to(device)
                optimizer.zero_grad()
                if training_method == 'contrastive_divergence':
                    self.model.contrastive_divergence(data1, data2)
                elif training_method == 'maximum_likelihood':
                    self.model.maximum_likelihood(data1, data2)
                optimizer.step()

                batch_loss = self.compute_loss()
                epoch_loss += batch_loss['mse_Wb']

            avg_epoch_loss = epoch_loss / len(self.dataloader)

            loss = self.compute_loss()
            loss.update({
                'learning_rate'  : optimizer.param_groups[0]['lr'],
                'time'           : time.time() - start_time,
                'epoch'          : epoch + 1,
                'avg_epoch_loss' : avg_epoch_loss
            })
            losses.append(loss)

            # Report training progress
            print(f"Epoch {epoch + 1}: Average loss = {avg_epoch_loss:.6f}, Time elapsed: {time.time() - start_time:.2f} seconds")

            epoch += 1

        return {"score": avg_epoch_loss}, losses

    def save_results(self, training_method, learning_rate, losses):
        """Save training results to a file."""
        results_filename = self.config.get_results_filename(training_method)

        results_to_save = {
            "data_parameters": {
                "n1" : self.config.n1,
                "n2" : self.config.n2,
                "data_size" : self.config.data_size,
                "batch_size" : self.config.batch_size
            },
            "train_parameters": {
                "training_method" : training_method,
                "time_limit" : self.config.time_limit,
                "cd_step" : self.config.cd_step
            },
            "learning_rate" : learning_rate,
            "losses" : losses
        }

        with open(results_filename, 'w') as f:
            json.dump(results_to_save, f, indent=2)

        files.download(results_filename)
        print(f"Training results saved and downloaded: {results_filename}")

def run_training(config, model, dataloader):
    # Trainer instance
    trainer = Trainer(config, model, dataloader)
    results = []

    for training_method in config.training_methods:
        for learning_rate in config.learning_rates:
            print(f"\nTraining with method: {training_method}, learning rate: {learning_rate}")

            _, losses = trainer.train(training_method, learning_rate)
            trainer.save_results(training_method, learning_rate, losses)

            results.append({
                'training_method' : training_method,
                'learning_rate'   : learning_rate,
                'losses'          : losses
            })

    return results

# Run training
results = run_training(config, model, dataloader)
print("Training completed for all methods and learning rates.")

## **Plotting Training Data**

We plot the training results.

#### **LaTeX Setting**

You can use the Palatino font (as in our preprint) if you would like to.

In [ ]:
use_latex = True  # True if you want to use the same font as in our paper.

# It may take a few minutes
if use_latex:
    ! sudo apt-get update
    ! sudo apt-get install texlive-latex-recommended
    ! sudo apt-get install dvipng texlive-latex-extra texlive-fonts-recommended texlive-fonts-extra
    ! wget http://mirrors.ctan.org/macros/latex/...
    ! unzip type1cm.zip -d /tmp/type1cm
    ! cd /tmp/type1cm/type1cm/ && sudo latex type1cm.ins
    ! sudo mkdir /usr/share/texmf/tex/latex/type1cm
    ! sudo cp /tmp/type1cm/type1cm/type1cm.sty /usr/share/texmf/tex/latex/type1cm
    ! sudo texhash
    !apt install cm-super
    plt.rcParams.update({
        "text.usetex": True,
        "font.family": "palatino",
        "font.serif": ["Palatino"],
    })

#### **Plotting Training Data**

In [ ]:
from matplotlib.ticker import LogLocator, FixedLocator, FixedFormatter

class Visualizer:
    def __init__(self, config):
        self.config = config

    @staticmethod
    def plot_progress(results, ax):
        """Plot training progress for a single run."""
        times = [entry['time'] for entry in results['losses']]
        errors = [entry['mse_Wb'] for entry in results['losses']]
        training_method = results['train_parameters']['training_method']

        linestyle = '-' if training_method == 'contrastive_divergence' else '--'
        color = 'blue' if training_method == 'contrastive_divergence' else 'red'
        training_method_name = 'Contrastive Divergence' if training_method == 'contrastive_divergence' else 'Maximum Likelihood'

        line, = ax.plot(times, errors, linestyle=linestyle, color=color, linewidth=3)

        return line, training_method_name

    def visualize_results(self, download):
        """Visualize results from JSON files and save as PDF."""
        size = 9
        figsize = (1.618 * size, size)
        fontsize = 36

        fig, ax = plt.subplots(figsize=figsize)

        # Find all result files
        result_files = glob.glob("training_data_*.json")

        if not result_files:
            print("No result files found.")
            return

        lines = []
        labels = []
        all_errors = []

        for file in result_files:
            with open(file, 'r') as f:
                results = json.load(f)

            line, label = self.plot_progress(results, ax)
            lines.append(line)
            labels.append(label)
            all_errors.extend([entry['mse_Wb'] for entry in results['losses']])

        # Legend
        ax.legend(lines, labels, fontsize=fontsize, frameon=False, loc='upper right', bbox_to_anchor=(1, 1))

        # Axes Label
        ax.set_xlabel(r'Time (minutes)', fontsize=fontsize)
        ax.set_ylabel(r'Mean Squared Error', fontsize=fontsize)

        # x-axis Ticks
        max_minutes = self.config.time_limit // 60
        tick_interval = 30  # 30 minutes
        tick_locations = range(0, max_minutes + 1, tick_interval)
        tick_labels = [str(t) for t in tick_locations]
        ax.set_xticks([t * 60 for t in tick_locations])  # Convert minutes to seconds for actual tick positions
        ax.set_xticklabels(tick_labels)

        # y-axis Ticks
        ax.set_yscale('log') # log scale
        y_min, y_max = min(all_errors), max(all_errors)
        ax.set_ylim(y_min * 0.9, y_max * 1.2)  # Set y limits with some padding
        y_ticks = [0.01, 0.1, 0.5, 1]
        y_labels = ['0.01', '0.1', '0.5', '1']
        ax.yaxis.set_major_locator(FixedLocator(y_ticks))
        ax.yaxis.set_major_formatter(FixedFormatter(y_labels))

        # Add minor ticks
        ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=10))
        ax.tick_params(axis='y', which='minor', left=True, labelleft=False)

        ax.tick_params(axis='both', which='major', labelsize=fontsize, colors='k')
        ax.tick_params(axis='both', which='minor', labelsize=fontsize * 0.8, colors='k')

        # Display the plot in Colab
        display(fig)

        # Save as PDF
        if download == True:
            matplotlib.use('PDF')  # To render the downloaded figure a PDF file
            pdf_filename = 'training_progress.pdf'
            plt.savefig(pdf_filename, format='pdf', dpi=300, bbox_inches='tight')

            # Close the figure to free up memory
            plt.close(fig)

            # Download the PDF file
            files.download(pdf_filename)

            print(f"Plotted results from {len(result_files)} files and saved as {pdf_filename}")
            for file in result_files:
                print(f"  {file}")

visualizer = Visualizer(config)
visualizer.visualize_results(download=False)